# Notebook 07 — Domain Gap Analysis (GPU — RunPod)

**Purpose:** Quantify how well Earth-pretrained features transfer to Titan SAR.

**Experiments:**
1. Random initialisation vs ImageNet pretrained vs SAR-specific pretrained
2. Frozen encoder (linear probing) to measure raw feature quality
3. Feature map visualisation

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else '.')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils import MODELS_DIR, FIGURES_DIR, PROCESSED_DIR, get_logger

log = get_logger('07_domain_gap')

## GPU Training Commands

Run these on RunPod with the same dataset archive from NB06:

```bash
# Experiment 1: Random init
python src/train.py --config configs/unet_resnet34.yaml \
    --run-name domain_gap_random
# (modify config to set encoder_weights: null before running)

# Experiment 2: ImageNet pretrained (same as Run 1 from NB06)
# Already done as unet_r34_focal

# Experiment 3: Frozen encoder (linear probing)
# Use a modified train.py that freezes encoder parameters
python src/train.py --config configs/unet_resnet34.yaml \
    --run-name domain_gap_frozen_encoder
# (modify script to add: for p in model.encoder.parameters(): p.requires_grad = False)
```

## Post-Training Analysis

Run after GPU experiments complete.

In [ ]:
# Load convergence logs from TensorBoard
# If TensorBoard logs are available, parse them; otherwise load from metric JSONs

experiments = {
    'Random Init': 'domain_gap_random',
    'ImageNet Pretrained': 'unet_r34_focal',
    'Frozen Encoder': 'domain_gap_frozen_encoder',
}

results = {}
for label, run_name in experiments.items():
    mpath = MODELS_DIR / f'{run_name}_metrics.json'
    if mpath.exists():
        with open(mpath) as f:
            results[label] = json.load(f)
        print(f"{label}: test_iou = {results[label].get('test_iou', 'N/A')}")
    else:
        print(f"{label}: metrics not found ({mpath})")

In [ ]:
# Try to load TensorBoard event files for convergence curves
try:
    from tensorboard.backend.event_processing import event_accumulator
    
    convergence_data = {}
    for label, run_name in experiments.items():
        log_dir = Path(f'runs/{run_name}')
        if not log_dir.exists():
            log_dir = Path(f'../runs/{run_name}')
        if log_dir.exists():
            ea = event_accumulator.EventAccumulator(str(log_dir))
            ea.Reload()
            if 'mIoU/val' in ea.Tags()['scalars']:
                events = ea.Scalars('mIoU/val')
                epochs = [e.step for e in events]
                values = [e.value for e in events]
                convergence_data[label] = (epochs, values)
    
    if convergence_data:
        fig, ax = plt.subplots(figsize=(10, 6))
        for label, (epochs, values) in convergence_data.items():
            ax.plot(epochs, values, label=label, linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation mIoU')
        ax.set_title('Domain Gap: Convergence Comparison')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'domain_gap_convergence.png', dpi=200, bbox_inches='tight')
        plt.show()
    else:
        print("No TensorBoard logs found. Plot convergence from logged CSVs if available.")
        
except ImportError:
    print("tensorboard not installed. Install to visualise convergence curves.")

In [ ]:
# Summary comparison table
if results:
    summary = []
    for label, data in results.items():
        summary.append({
            'Initialisation': label,
            'Best Val mIoU': data.get('best_val_iou', 'N/A'),
            'Test mIoU': data.get('test_iou', 'N/A'),
        })
    df = pd.DataFrame(summary)
    print(df.to_string(index=False))
    
    # Save
    df.to_csv(MODELS_DIR / 'domain_gap_comparison.csv', index=False)
else:
    print("Run GPU experiments first.")

## Feature Map Visualisation

Visualise intermediate feature maps from different encoders on the same Titan SAR tile.

In [ ]:
# This requires GPU and PyTorch
try:
    import torch
    import segmentation_models_pytorch as smp
    
    if torch.cuda.is_available():
        device = torch.device('cuda')
    else:
        device = torch.device('cpu')
        print("Running on CPU — feature vis will be slow but works.")
    
    # Load a sample tile
    sample_tile = np.load(PROCESSED_DIR / 'sar_tiles' / sorted(os.listdir(PROCESSED_DIR / 'sar_tiles'))[0])
    x = torch.from_numpy(sample_tile).float().unsqueeze(0).unsqueeze(0).expand(1, 3, -1, -1).to(device)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    for row, (label, run_name) in enumerate(list(experiments.items())[:2]):
        weight_path = MODELS_DIR / f'{run_name}_best.pth'
        if not weight_path.exists():
            for ax in axes[row]:
                ax.text(0.5, 0.5, f'{label}\n(weights not found)', 
                       ha='center', va='center', transform=ax.transAxes)
                ax.axis('off')
            continue
        
        model = smp.Unet('resnet34', in_channels=3, classes=6).to(device)
        model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
        model.eval()
        
        # Extract encoder features at different stages
        features = model.encoder(x)
        
        for col, (feat, stage) in enumerate(zip(features[:4], ['Stage 0', 'Stage 1', 'Stage 2', 'Stage 3'])):
            feat_map = feat[0].mean(dim=0).detach().cpu().numpy()
            axes[row, col].imshow(feat_map, cmap='viridis')
            axes[row, col].set_title(f'{label}\n{stage} ({feat.shape[1]}ch)', fontsize=9)
            axes[row, col].axis('off')
    
    plt.suptitle('Encoder Feature Maps on Titan SAR', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'feature_map_visualisation.png', dpi=200, bbox_inches='tight')
    plt.show()

except ImportError:
    print("PyTorch/smp not available. Run this cell on GPU.")